In [0]:
#-------------------------------------------------------------------------------------------#
# Data criacao: 15/11/2025                                                                  #
# Objetivo:                                                                                 #
#   Criar uma tabela de ordens silver ja com:                                               #  
#   Quantidade de itens, vendendores, lojas, mapeamento de Status e valor total da venda    #
#   (valor total - desconto)                                                                #
#    "Mapeamento Status:"                                                                   #
#    "1 THEN 'Pending',"                                                                    #
#    "2 THEN 'Processing',"                                                                 #
#    "3 THEN 'Shipped',"                                                                    #
#    "4 THEN 'Delivered',"                                                                  #
#    "ELSE 'Unknown'"                                                                       #
#    Origem camada bronze: orders                                                           #
#                          stores                                                           #
#                          staffs                                                           #
#                          order_items                                                      #
#    Destino camada silver: orders                                                          #
#-------------------------------------------------------------------------------------------#
# 1. Criar uma tabela temporária com os dados da tabela bronze                              #
# 2. Criar uma tabela Silver com os dados da tabela bronze                                  #
# 4. Importar as funções desejadas                                                          #
#-------------------------------------------------------------------------------------------#


In [0]:
import sys
import importlib

# 1. Garante que alterações no arquivo .py sejam carregadas automaticamente
%load_ext autoreload
%autoreload 2

# 2. Adiciona a pasta pai ('Codigo') com o prefixo '/Workspace'
caminho_codigo = '/Workspace/Users/elisangela.soares.santos83@gmail.com/Projeto_Azure_Databricks/Codigo'
if caminho_codigo not in sys.path:
    sys.path.append(caminho_codigo)

# 3. Importa e força o recarregamento do módulo
import funcoes_diversas.funcoes_util
importlib.reload(funcoes_diversas.funcoes_util)

# 4. Importa as funções desejadas
from funcoes_diversas.funcoes_util import buscar_caminho, gerar_tabela_temporaria

In [0]:
mapa_criado = gerar_tabela_temporaria(["orders", "stores","staffs","order_items"])

In [0]:
df_orders_silver = spark.sql("""with order_items as(select OI.order_id, 
       OI.item_id,
       OI.product_id, 
       OI.quantity,
       OI.list_price,
       round((OI.list_price * OI.quantity) *(1-OI.discount),2) as total_sale,
       OI.discount
    from tmp_bronze_order_items as OI)
    select ORD.order_id,
           ORD.customer_id,
           ORD.order_status,
           CASE ORD.order_status
                WHEN 1 THEN 'Pending'
                WHEN 2 THEN 'Processing'
                WHEN 3 THEN 'Shipped'
                WHEN 4 THEN 'Delivered'
                       ELSE 'Unknown' END AS status_descricao,
           ORD.order_date,
           ORD.required_date,
           ORD.shipped_date,
           ORD.staff_id,
           ST.store_name,
           ST.city,
           ST.state,
           STF.first_name,
           STF.email,
           STF.active ,  
           OI.product_id,
           OI.quantity,
           OI.total_sale,
           OI.list_price,
           OI.discount
    FROM tmp_bronze_orders      as ORD 
    LEFT JOIN tmp_bronze_stores as ST  on ORD.store_id  = ST.store_id
    LEFT JOIN tmp_bronze_staffs as STF on ORD.staff_id  = STF.staff_id
    LEFT JOIN order_items       as OI  on ORD.order_id  = OI.order_id
     """)

In [0]:
silver_path = buscar_caminho("silver")
# transformando consulta em uma tabela Silver orders 
df_orders_silver.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .save(f"{silver_path}/orders")